# Day 3 · Notebook 1 — Talking to our model over HTTP

On Day 2 we produced `models/model_fp32.onnx` and `models/model_int8.onnx`.
Today the model leaves the notebook and becomes a **service**:

```
 ┌──────────────┐   HTTP (JSON / multipart)   ┌──────────────────────────┐
 │  Client      │ ──────────────────────────▶ │  Flask API (app/api)     │
 │  notebook,   │                              │   preprocess → ONNX RT   │
 │  curl,       │ ◀────────────────────────── │   → softmax → JSON       │
 │  Streamlit   │        {"prediction": ...}  └──────────────────────────┘
 └──────────────┘
```

In this notebook we act as the **client**: start the API, call every endpoint, look at errors, and measure latency.

> **Run this notebook from the `Day3_Serving_Flask_Streamlit` folder.**

## 1. Start the API

In class we normally run the API in a **separate terminal** so we can watch its logs:

```bash
cd Day3_Serving_Flask_Streamlit/app/api
# Windows
waitress-serve --port=5000 app:app
# Linux / macOS
gunicorn -w 2 -b 0.0.0.0:5000 app:app
```

For convenience, the next cell starts it **from the notebook** (using Flask's dev server) if nothing is listening on port 5000 yet.

In [ ]:
import subprocess, sys, time, os
from pathlib import Path
import requests

API = "http://127.0.0.1:5000"   # not "localhost": on Windows it resolves to IPv6 ::1 first -> +2 s per request
API_DIR = Path("app/api").resolve()

def api_up():
    try:
        return requests.get(f"{API}/health", timeout=1).status_code == 200
    except requests.ConnectionError:
        return False

server = None
if not api_up():
    # Uses the SAME Python as this notebook kernel -> select the course .venv kernel!
    log_file = open("api_server.log", "w")
    server = subprocess.Popen([sys.executable, "app.py"], cwd=API_DIR,
                              stdout=log_file, stderr=subprocess.STDOUT)
    for _ in range(60):          # wait up to ~30 s for models to load
        if api_up() or server.poll() is not None:
            break
        time.sleep(0.5)
    if not api_up():
        log_file.close()
        raise RuntimeError("API failed to start. Server log:\n" + Path("api_server.log").read_text()
                           + f"\nKernel Python: {sys.executable} - is this the course venv?")
print("API up:", api_up(), "| started by notebook:", server is not None)

## 2. Health vs readiness

Two different questions — Kubernetes asks both on Day 4:

| Endpoint | Question | If it fails… |
|---|---|---|
| `/health` (liveness) | Is the process alive? | k8s **restarts** the container |
| `/ready` (readiness) | Can it serve traffic (models loaded)? | k8s **stops sending traffic** to it, no restart |

💬 *Discussion:* why would it be a bad idea to check "model loaded" in the liveness probe?

In [ ]:
for path in ["/", "/health", "/ready"]:
    r = requests.get(API + path)
    print(f"GET {path:8s} -> {r.status_code}  {r.json()}")

In [ ]:
info = requests.get(f"{API}/models").json()
import pandas as pd
pd.DataFrame(info["variants"]).T

## 3. Predict — multipart upload

This is what `curl -F file=@cat.png "http://localhost:5000/predict?model=int8"` does.
We use a sample CIFAR-10 test image from `app/ui/samples/`.

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

samples = sorted(Path("app/ui/samples").glob("*.png"))
print(len(samples), "sample images:", [p.stem for p in samples][:6], "...")
img_path = samples[6] if len(samples) > 6 else samples[0]

with open(img_path, "rb") as f:
    r = requests.post(f"{API}/predict", params={"model": "int8", "top_k": 3},
                      files={"file": (img_path.name, f, "image/png")})
res = r.json()
plt.figure(figsize=(3, 3)); plt.imshow(Image.open(img_path)); plt.axis("off")
plt.title(f"true: {img_path.stem.split('_')[0]}  |  pred: {res['prediction']} ({res['confidence']:.0%})")
plt.show()
res

### Reading the response
* `inference_ms` — only the ONNX Runtime call.
* `total_ms` — server-side time including decoding + preprocessing.
* Client-side time (below) adds network + HTTP overhead.

Knowing **where** time goes is the first step of any optimization.

In [ ]:
t0 = time.perf_counter()
r = requests.post(f"{API}/predict", files={"file": img_path.read_bytes()})
client_ms = (time.perf_counter() - t0) * 1000
j = r.json()
print(f"inference {j['inference_ms']:.2f} ms | server total {j['total_ms']:.2f} ms | client round-trip {client_ms:.2f} ms")

## 4. Predict — JSON with base64

Browsers and mobile apps often send JSON instead of multipart. Base64 inflates the payload by ~33 %.

In [ ]:
import base64
b64 = base64.b64encode(img_path.read_bytes()).decode()
print("raw bytes:", img_path.stat().st_size, "| base64 chars:", len(b64))
requests.post(f"{API}/predict", json={"image_b64": b64}).json()["top_k"]

## 5. Errors are part of the API contract

A good API fails **loudly and clearly** with the right status code:
`400` bad input · `404` unknown resource · `413` too large · `5xx` server bug.

In [ ]:
cases = {
    "no file":          dict(),
    "not an image":     dict(files={"file": ("x.png", b"hello", "image/png")}),
    "bad base64":       dict(json={"image_b64": "%%%"}),
    "unknown model":    dict(params={"model": "fp64"}, files={"file": img_path.read_bytes()}),
    "too large (6 MB)": dict(files={"file": ("big.png", os.urandom(6 * 1024 * 1024), "image/png")}),
}
for name, kw in cases.items():
    r = requests.post(f"{API}/predict", **kw)
    print(f"{name:18s} -> {r.status_code}  {r.json()}")

## 6. fp32 vs int8 — what difference does quantization make *in the service*?

`/compare` runs the same image through every variant. Let's run it over all sample images.

In [ ]:
rows = []
for p in samples:
    out = requests.post(f"{API}/compare", files={"file": p.read_bytes()}).json()
    row = {"image": p.stem, "true": p.stem.split("_")[0], "agree": out["agree"]}
    for v, r_ in out["results"].items():
        row[f"{v}_pred"] = r_["prediction"]
        row[f"{v}_conf"] = r_["confidence"]
        row[f"{v}_ms"] = r_["inference_ms"]
    rows.append(row)
cmp_df = pd.DataFrame(rows)
cmp_df

In [ ]:
summary = {}
for v in info["variants"]:
    summary[v] = {
        "accuracy_on_samples": (cmp_df[f"{v}_pred"] == cmp_df["true"]).mean(),
        "median_inference_ms": cmp_df[f"{v}_ms"].median(),   # median: robust to outliers
        "size_mb": info["variants"][v]["size_mb"],
    }
print("variants agree on", cmp_df["agree"].mean() * 100, "% of samples")
pd.DataFrame(summary).T

💬 *Discussion:* the int8 model is ~4× smaller. Is the **service** 4× faster? Why not?
(Hint: compare `inference_ms` with the client round-trip time — what else is in the request path?)

## 7. Load test: latency percentiles and throughput

Averages hide pain. Users feel the **tail** (p95/p99). We fire concurrent requests with a thread pool.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
import numpy as np

payload = img_path.read_bytes()

def call(model):
    t0 = time.perf_counter()
    requests.post(f"{API}/predict", params={"model": model}, files={"file": payload}).raise_for_status()
    return (time.perf_counter() - t0) * 1000

def load_test(model, n=100, concurrency=4):
    t0 = time.perf_counter()
    with ThreadPoolExecutor(concurrency) as pool:
        lat = np.array(list(pool.map(call, [model] * n)))
    wall = time.perf_counter() - t0
    return {"model": model, "concurrency": concurrency, "p50_ms": np.percentile(lat, 50),
            "p95_ms": np.percentile(lat, 95), "p99_ms": np.percentile(lat, 99), "rps": n / wall}

results = [load_test(m, n=100, concurrency=c) for m in info["variants"] for c in (1, 4)]
bench = pd.DataFrame(results).round(1)
bench

In [ ]:
ax = bench.pivot(index="concurrency", columns="model", values="p95_ms").plot.bar(figsize=(6, 3), rot=0)
ax.set_ylabel("p95 latency (ms)"); ax.set_title("Tail latency vs concurrency"); plt.show()

💬 *Discussion:*
* The Flask **dev server** (used here) handles requests with threads in one process. What changes with `gunicorn -w 4`?
* Why does latency increase with concurrency even though each request is "the same work"?
* Where is the bottleneck — CPU, Python GIL, network, preprocessing?

## 8. The #1 production bug: preprocessing mismatch

The API expects **RGB**. OpenCV loads images as **BGR**. What happens if a client "helpfully" preprocesses the wrong way?

In [ ]:
import io
def to_bytes(im):
    b = io.BytesIO(); im.save(b, format="PNG"); return b.getvalue()

rgb = Image.open(img_path).convert("RGB")
bgr = Image.fromarray(np.asarray(rgb)[:, :, ::-1].copy())     # swap channels like cv2.imread would
for name, im in [("RGB (correct)", rgb), ("BGR (wrong)", bgr)]:
    j = requests.post(f"{API}/predict", files={"file": to_bytes(im)}).json()
    print(f"{name:14s} -> {j['prediction']:10s} {j['confidence']:.2%}")

In [ ]:
wrong = 0
for p in samples:
    im = Image.open(p).convert("RGB")
    swapped = Image.fromarray(np.asarray(im)[:, :, ::-1].copy())
    a = requests.post(f"{API}/predict", files={"file": to_bytes(im)}).json()["prediction"]
    b = requests.post(f"{API}/predict", files={"file": to_bytes(swapped)}).json()["prediction"]
    wrong += a != b
print(f"Channel swap changed the prediction on {wrong}/{len(samples)} images")

The service never crashed and never raised an error — it just silently got worse.
**This is why the preprocessing lives inside the API**, not in each client.

## 9. Out-of-distribution input

The model only knows 10 classes. What does it say about something that is none of them?

In [ ]:
noise = Image.fromarray((np.random.default_rng(0).random((96, 96, 3)) * 255).astype("uint8"))
blank = Image.new("RGB", (96, 96), (255, 255, 255))
for name, im in [("random noise", noise), ("white image", blank)]:
    j = requests.post(f"{API}/predict", files={"file": to_bytes(im)}).json()
    print(f"{name:13s} -> {j['prediction']:10s} confidence {j['confidence']:.2%}")

💬 *Discussion:* softmax **always** sums to 1, so the model always picks *something*.
How could we detect "I don't know"? (confidence threshold, entropy, an extra "other" class, OOD detectors…)

## 10. Server metrics & cleanup

In [ ]:
requests.get(f"{API}/metrics").json()

In [ ]:
if server is not None:
    server.terminate(); server.wait(timeout=10)
    print("API stopped")

## ✅ Recap
* A model becomes useful when it's behind a **stable contract** (endpoints, input format, status codes).
* Keep preprocessing **server-side** and identical to training.
* Measure latency at three levels: inference · server · client. Report **p95**, not the mean.
* Next: the Streamlit UI (`app/ui`) is just another client of this API. On Day 4 we put both into containers.